# Aplicação do modelo de bigramas

O modelo estima `P(token atual | token anterior)`, calcula perplexidade no conjunto de teste e gera uma pequena sequência de texto.


In [ ]:
from pathlib import Path
import re
import sys

import tiktoken

def localizar_raiz() -> Path:
    atual = Path.cwd().resolve()
    for candidato in [atual, *atual.parents]:
        if (candidato / "02-modelo-bigrama").exists():
            return candidato
    raise FileNotFoundError("Não foi possível localizar a raiz do repositório.")

ROOT = localizar_raiz()
MODULO = ROOT / "02-modelo-bigrama"
sys.path.insert(0, str(MODULO))

from bigram_model import BigramModel


In [ ]:
caminho_texto = ROOT / "dados" / "texto_bigrama.txt"

if caminho_texto.exists():
    texto = caminho_texto.read_text(encoding="utf-8")
else:
    texto = (
        "Modelos de linguagem aprendem padrões presentes em sequências de texto. "
        "Um modelo de bigramas observa pares consecutivos de tokens. "
        "A probabilidade do próximo token depende do token atual. "
        "Modelos estatísticos simples ajudam a entender conceitos usados em modelos maiores. "
        "A perplexidade mede o quanto o modelo se surpreende com uma sequência. "
        "Quanto menor a perplexidade, melhor o ajuste do modelo aos dados avaliados."
    )

sentencas = [s.strip() for s in re.split(r"(?<=[.!?])\s+", texto) if s.strip()]
print(f"Sentenças disponíveis: {len(sentencas)}")


In [ ]:
tokenizer = tiktoken.get_encoding("gpt2")
model = BigramModel(tokenizer=tokenizer, alpha=1.0, seed=42)

train_tokens, test_tokens = model.prepare_data(sentencas, train_ratio=0.8)
model.train(train_tokens)

print(f"Tokens de treino: {len(train_tokens)}")
print(f"Tokens de teste: {len(test_tokens)}")
print(f"Vocabulário observado: {len(model.vocab)}")


In [ ]:
perplexidade = model.calculate_perplexity(test_tokens)
print(f"Perplexidade no conjunto de teste: {perplexidade:.2f}")


In [ ]:
texto_gerado = model.generate_text(max_tokens=40)
print("Texto gerado:")
print(texto_gerado)


## Interpretação

Bigramas são úteis para visualizar a ideia de dependência entre tokens, mas possuem contexto muito limitado. A perplexidade e a qualidade da geração tendem a melhorar com mais dados, enquanto modelos modernos capturam dependências muito mais longas.
